# SeaAlert - Notebook 03: Noise Augmentation and ASR

This notebook adds radio-like noise to clean audio and transcribes using Whisper.

## Cell 0 - Setup

In [1]:
# Mount Google Drive (Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Install dependencies
!pip install -q numpy pandas tqdm soundfile scipy faster-whisper

# Imports
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal
from tqdm import tqdm

# Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Project directory
if IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive/SeaAlert")
else:
    PROJECT_DIR = Path(".").resolve().parent

# Create folders
AUDIO_NOISY_DIR = PROJECT_DIR / "data" / "audio_noisy"
ASR_DIR = PROJECT_DIR / "data" / "asr"
RESULTS_DIR = PROJECT_DIR / "results"

for level in ["low", "med", "high"]:
    (AUDIO_NOISY_DIR / level).mkdir(parents=True, exist_ok=True)
ASR_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
SAMPLE_RATE = 16000
SNR_LEVELS = {"low": 18, "med": 12, "high": 6}  # dB
MAX_SAMPLES = None  # Set integer for quick run
CHECKPOINT_EVERY = 50

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"SNR Levels: {SNR_LEVELS}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.3 MB/s eta 0:00:00
PROJECT_DIR: /content/drive/MyDrive/SeaAlert
SNR Levels: {'low': 18, 'med': 12, 'high': 6}


## Cell 1 - Load Audio Index

In [2]:
AUDIO_INDEX_PATH = PROJECT_DIR / "data" / "audio_clean" / "audio_index.csv"

if not AUDIO_INDEX_PATH.exists():
    raise FileNotFoundError(f"Audio index not found: {AUDIO_INDEX_PATH}. Run notebook 02 first.")

audio_df = pd.read_csv(AUDIO_INDEX_PATH)
print(f"Loaded {len(audio_df)} audio file references")

# Verify files exist
valid_mask = audio_df["wav_path_clean"].apply(lambda x: Path(x).exists())
print(f"Valid files: {valid_mask.sum()}/{len(audio_df)}")

audio_df = audio_df[valid_mask].reset_index(drop=True)

# Subset for quick run
if MAX_SAMPLES is not None:
    audio_df = audio_df.head(MAX_SAMPLES)
    print(f"Subset to {len(audio_df)} samples")

audio_df.head()

Loaded 1872 audio file references
Valid files: 1872/1872


,idx,wav_path_clean,label,style,scenario_type,speaker
0,0,/content/drive/MyDrive/SeaAlert/data/audio_cle...,Distress,formal,nav_hazard,NaN
1,1,/content/drive/MyDrive/SeaAlert/data/audio_cle...,Safety,formal,medical_issue,NaN
2,2,/content/drive/MyDrive/SeaAlert/data/audio_cle...,Safety,third_party,steering_failure,NaN
3,3,/content/drive/MyDrive/SeaAlert/data/audio_cle...,Distress,formal,medical_issue,NaN
4,4,/content/drive/MyDrive/SeaAlert/data/audio_cle...,Distress,formal,water_ingress,NaN


## Cell 2 - Noise Augmentation Functions

In [3]:
def generate_bandpass_noise(length, sample_rate=SAMPLE_RATE,
                            low_freq=300, high_freq=3400, amplitude=0.01):
    """Generate band-limited noise simulating radio frequency response."""
    noise = np.random.normal(0, amplitude * 3, length)

    nyquist = sample_rate / 2
    low = max(0.01, min(low_freq / nyquist, 0.99))
    high = max(low + 0.01, min(high_freq / nyquist, 0.99))

    try:
        b, a = signal.butter(4, [low, high], btype='band')
        filtered = signal.filtfilt(b, a, noise)
        return filtered
    except:
        return noise

def generate_static_bursts(length, num_bursts=5, burst_duration_range=(100, 500), amplitude=0.3):
    """Generate random static bursts."""
    result = np.zeros(length)

    for _ in range(num_bursts):
        burst_len = random.randint(*burst_duration_range)
        burst_start = random.randint(0, max(0, length - burst_len))

        burst = np.random.uniform(-amplitude, amplitude, burst_len)
        envelope = np.hanning(burst_len)
        burst = burst * envelope

        result[burst_start:burst_start + burst_len] += burst

    return result

def generate_dropouts(length, num_dropouts=3, dropout_duration_range=(50, 200)):
    """Generate dropout mask."""
    mask = np.ones(length)

    for _ in range(num_dropouts):
        dropout_len = random.randint(*dropout_duration_range)
        dropout_start = random.randint(0, max(0, length - dropout_len))

        fade_len = min(20, dropout_len // 4)
        mask[dropout_start:dropout_start + fade_len] *= np.linspace(1, 0, fade_len)
        mask[dropout_start + fade_len:dropout_start + dropout_len - fade_len] = 0
        if dropout_len > 2 * fade_len:
            end_start = dropout_start + dropout_len - fade_len
            mask[end_start:dropout_start + dropout_len] *= np.linspace(0, 1, fade_len)

    return mask

def generate_hum(length, sample_rate=SAMPLE_RATE, frequency=50.0, amplitude=0.005):
    """Generate power line hum."""
    t = np.arange(length) / sample_rate
    hum = amplitude * np.sin(2 * np.pi * frequency * t)
    hum += (amplitude * 0.5) * np.sin(2 * np.pi * frequency * 2 * t)
    return hum

def apply_snr(clean_audio, noise, target_snr_db):
    """Mix noise with audio at specified SNR."""
    signal_rms = np.sqrt(np.mean(clean_audio ** 2))
    noise_rms = np.sqrt(np.mean(noise ** 2))

    if noise_rms < 1e-10:
        return clean_audio

    target_noise_rms = signal_rms / (10 ** (target_snr_db / 20))
    noise_scaled = noise * (target_noise_rms / noise_rms)

    noisy = clean_audio + noise_scaled

    max_val = np.max(np.abs(noisy))
    if max_val > 1.0:
        noisy = noisy / max_val * 0.95

    return noisy

def add_radio_noise(audio, noise_level="med"):
    """Add composite radio-like noise."""
    length = len(audio)

    params = {
        "low": {"noise_amp": 0.005, "num_bursts": 2, "burst_amp": 0.1,
                "num_dropouts": 1, "hum_amp": 0.002},
        "med": {"noise_amp": 0.01, "num_bursts": 4, "burst_amp": 0.2,
                "num_dropouts": 2, "hum_amp": 0.004},
        "high": {"noise_amp": 0.02, "num_bursts": 8, "burst_amp": 0.35,
                 "num_dropouts": 4, "hum_amp": 0.008},
    }

    p = params.get(noise_level, params["med"])
    target_snr = SNR_LEVELS[noise_level]

    # Generate noise components
    noise = generate_bandpass_noise(length, amplitude=p["noise_amp"])
    bursts = generate_static_bursts(length, p["num_bursts"], amplitude=p["burst_amp"])
    hum = generate_hum(length, amplitude=p["hum_amp"])

    composite_noise = noise + bursts + hum

    # Apply SNR
    noisy = apply_snr(audio, composite_noise, target_snr)

    # Apply dropouts
    dropout_mask = generate_dropouts(length, p["num_dropouts"])
    noisy = noisy * dropout_mask

    # Apply clipping for heavier noise
    if noise_level in ["med", "high"]:
        threshold = 0.9 if noise_level == "med" else 0.8
        max_val = np.max(np.abs(noisy)) * threshold
        noisy = np.clip(noisy, -max_val, max_val)

    return noisy

print("Noise functions loaded.")

Noise functions loaded.


## Cell 3 - Generate Noisy Audio

In [4]:
NOISY_INDEX_PATH = AUDIO_NOISY_DIR / "noisy_index.csv"
LEVELS = ["low", "med", "high"]

# Check existing progress
existing_indices = set()
results = []

if NOISY_INDEX_PATH.exists():
    existing_df = pd.read_csv(NOISY_INDEX_PATH)
    existing_indices = set(existing_df["idx"].tolist())
    results = existing_df.to_dict('records')
    print(f"Resuming: {len(results)} files already processed")

# Process audio files
pbar = tqdm(audio_df.iterrows(), total=len(audio_df), desc="Adding noise")

for i, row in pbar:
    idx = row["idx"]

    if idx in existing_indices:
        continue

    clean_path = Path(row["wav_path_clean"])

    try:
        audio, sr = sf.read(str(clean_path))
    except Exception as e:
        print(f"\nError loading {clean_path}: {e}")
        continue

    result_row = {
        "idx": idx,
        "wav_clean": str(clean_path),
    }

    # Generate noisy versions
    for level in LEVELS:
        noisy = add_radio_noise(audio, level)
        out_path = AUDIO_NOISY_DIR / level / f"{idx}.wav"
        sf.write(str(out_path), noisy, sr)
        result_row[f"wav_{level}"] = str(out_path)

    results.append(result_row)

    # Checkpoint
    if len(results) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(NOISY_INDEX_PATH, index=False)

# Final save
noisy_df = pd.DataFrame(results)
noisy_df.to_csv(NOISY_INDEX_PATH, index=False)

print(f"\nGenerated noisy audio for {len(results)} samples")
print(f"Index saved to: {NOISY_INDEX_PATH}")

Adding noise: 100%|██████████| 1872/1872 [35:03<00:00,  1.12s/it]


Generated noisy audio for 1872 samples
Index saved to: /content/drive/MyDrive/SeaAlert/data/audio_noisy/noisy_index.csv


## Cell 4 - ASR Transcription

In [5]:
from faster_whisper import WhisperModel

# Select model size
WHISPER_MODEL = "base"  # tiny, base, small, medium, large

# Device selection
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"Loading Whisper model: {WHISPER_MODEL} on {device}")
whisper_model = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute_type)
print("Whisper model loaded")

def transcribe_audio(audio_path):
    """Transcribe audio file using Whisper."""
    try:
        segments, info = whisper_model.transcribe(
            str(audio_path),
            language="en",
            beam_size=5,
        )
        text = " ".join([seg.text for seg in segments])
        return text.strip()
    except Exception as e:
        print(f"Transcription failed: {e}")
        return ""

Loading Whisper model: base on cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocabulary.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/145M [00:00<?, ?B/s]

Whisper model loaded


In [6]:
ASR_TRANSCRIPTS_PATH = ASR_DIR / "asr_transcripts.csv"
TRANSCRIBE_LEVELS = ["low", "med", "high"]  # Levels to transcribe

# Reload noisy index
noisy_df = pd.read_csv(NOISY_INDEX_PATH)

# Check existing progress
existing_indices = set()
transcript_results = []

if ASR_TRANSCRIPTS_PATH.exists():
    existing_df = pd.read_csv(ASR_TRANSCRIPTS_PATH)
    existing_indices = set(existing_df["idx"].tolist())
    transcript_results = existing_df.to_dict('records')
    print(f"Resuming: {len(transcript_results)} already transcribed")

# Transcribe
pbar = tqdm(noisy_df.iterrows(), total=len(noisy_df), desc="Transcribing")

for i, row in pbar:
    idx = row["idx"]

    if idx in existing_indices:
        continue

    result_row = {"idx": idx}

    for level in TRANSCRIBE_LEVELS:
        wav_col = f"wav_{level}"
        if wav_col not in row:
            result_row[f"asr_{level}"] = ""
            continue

        wav_path = row[wav_col]
        if not Path(wav_path).exists():
            result_row[f"asr_{level}"] = ""
            continue

        transcript = transcribe_audio(wav_path)
        result_row[f"asr_{level}"] = transcript

    transcript_results.append(result_row)

    # Checkpoint
    if len(transcript_results) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(transcript_results).to_csv(ASR_TRANSCRIPTS_PATH, index=False)

# Final save
transcripts_df = pd.DataFrame(transcript_results)
transcripts_df.to_csv(ASR_TRANSCRIPTS_PATH, index=False)

print(f"\nTranscribed {len(transcript_results)} samples")
print(f"Saved to: {ASR_TRANSCRIPTS_PATH}")

Transcribing: 100%|██████████| 1872/1872 [1:10:13<00:00,  2.25s/it]


Transcribed 1872 samples
Saved to: /content/drive/MyDrive/SeaAlert/data/asr/asr_transcripts.csv


## Cell 5 - WER Report

In [7]:
def calculate_wer(reference, hypothesis):
    """Calculate Word Error Rate."""
    ref_words = str(reference).lower().split()
    hyp_words = str(hypothesis).lower().split()

    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0

    m, n = len(ref_words), len(hyp_words)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_words[i-1] == hyp_words[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])

    return dp[m][n] / len(ref_words)

# Load original dataset for reference text
DATASET_PATH = PROJECT_DIR / "data" / "processed" / "02seaalert.csv"
original_df = pd.read_csv(DATASET_PATH)

# Merge with transcripts
transcripts_df = pd.read_csv(ASR_TRANSCRIPTS_PATH)
merged = pd.merge(original_df[["idx", "text"]], transcripts_df, on="idx")

# Compute WER
wer_results = []

for _, row in tqdm(merged.iterrows(), total=len(merged), desc="Computing WER"):
    ref = row["text"]
    result = {"idx": row["idx"]}

    for level in TRANSCRIBE_LEVELS:
        asr_col = f"asr_{level}"
        if asr_col in row:
            hyp = row[asr_col] if pd.notna(row[asr_col]) else ""
            wer = calculate_wer(ref, hyp)
            result[f"wer_{level}"] = wer

    wer_results.append(result)

wer_df = pd.DataFrame(wer_results)

# Summary
print("\n" + "=" * 40)
print("WER SUMMARY")
print("=" * 40)

for level in TRANSCRIBE_LEVELS:
    wer_col = f"wer_{level}"
    if wer_col in wer_df.columns:
        avg_wer = wer_df[wer_col].mean()
        print(f"{level.upper()}: Average WER = {avg_wer:.4f} ({avg_wer*100:.1f}%)")

# Save WER report
WER_REPORT_PATH = RESULTS_DIR / "wer_report.csv"
wer_df.to_csv(WER_REPORT_PATH, index=False)
print(f"\nWER report saved to: {WER_REPORT_PATH}")

Computing WER: 100%|██████████| 1872/1872 [00:11<00:00, 166.73it/s]


WER SUMMARY
LOW: Average WER = 0.2680 (26.8%)
MED: Average WER = 0.2965 (29.6%)
HIGH: Average WER = 0.3622 (36.2%)

WER report saved to: /content/drive/MyDrive/SeaAlert/results/wer_report.csv


## Cell 6 - Merge ASR into Dataset

In [8]:
# Load original dataset
original_df = pd.read_csv(DATASET_PATH)
transcripts_df = pd.read_csv(ASR_TRANSCRIPTS_PATH)

# Merge
merged_df = pd.merge(original_df, transcripts_df, on="idx", how="left")

# Save merged dataset
MERGED_PATH = PROJECT_DIR / "data" / "processed" / "03seaalert_with_asr.csv"
merged_df.to_csv(MERGED_PATH, index=False)

print(f"Merged dataset saved to: {MERGED_PATH}")
print(f"Shape: {merged_df.shape}")
print(f"\nColumns: {merged_df.columns.tolist()}")

# Sample comparison
print("\nSample ASR comparison:")
sample = merged_df.iloc[0]
print(f"Original: {sample['text'][:100]}...")
if 'asr_med' in sample and pd.notna(sample['asr_med']):
    print(f"ASR (med): {sample['asr_med'][:100]}...")

Merged dataset saved to: /content/drive/MyDrive/SeaAlert/data/processed/03seaalert_with_asr.csv
Shape: (1872, 19)

Columns: ['idx', 'text', 'label', 'style', 'scenario_type', 'has_codeword', 'codeword', 'text_masked', 'vessel', 'call_sign', 'mmsi', 'location', 'weather', 'pob', 'nature', 'injury', 'asr_low', 'asr_med', 'asr_high']

Sample ASR comparison:
Original: MAYDAY, MAYDAY, MAYDAY. This is the fishing vessel Ocean Hunter, call sign WXYZ123, MMSI 123456789. ...
ASR (med): Mayday. Mayday. Mayday.  This is the fishing vessel ocean hunter.  Call sign WXYZ-123.  MMMSI-123,45...


In [9]:
print("\n" + "=" * 60)
print("NOISE & ASR PROCESSING COMPLETE")
print("=" * 60)
print(f"\nArtifacts created:")
print(f"  1. {AUDIO_NOISY_DIR}/[low,med,high]/*.wav")
print(f"  2. {NOISY_INDEX_PATH}")
print(f"  3. {ASR_TRANSCRIPTS_PATH}")
print(f"  4. {WER_REPORT_PATH}")
print(f"  5. {MERGED_PATH}")


NOISE & ASR PROCESSING COMPLETE

Artifacts created:
  1. /content/drive/MyDrive/SeaAlert/data/audio_noisy/[low,med,high]/*.wav
  2. /content/drive/MyDrive/SeaAlert/data/audio_noisy/noisy_index.csv
  3. /content/drive/MyDrive/SeaAlert/data/asr/asr_transcripts.csv
  4. /content/drive/MyDrive/SeaAlert/results/wer_report.csv
  5. /content/drive/MyDrive/SeaAlert/data/processed/03seaalert_with_asr.csv
